# Relative Archie water-content change estimate

This notebook is a fast post-processing bridge between temperature-corrected time-lapse resistivity and the future AD-based water-content inversion.

It uses the relative Archie relation:

`theta(t) = theta0 * exp(-(log rho(t) - log rho0) / n)`

and compares estimated ERT water-content changes against TMC sensor water-content changes.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 120,
})
np.set_printoptions(precision=4, suppress=True)


In [ ]:
# Resolve paths and controls.
here = Path.cwd().resolve()
candidates = [here, *here.parents]
project_root = next((p for p in candidates if (p / 'adtlert').exists() and (p / 'ProcessedData').exists()), None)
if project_root is None:
    raise FileNotFoundError('Cannot locate project root containing adtlert/ and ProcessedData/.')

result_dir = project_root / 'result' / '8_real_data' / '2_timelapse_inversion_real_adtlert'
if not result_dir.exists():
    raise FileNotFoundError(f'Missing real-data result directory: {result_dir}')

save_figures = True
figure_dpi = 600

saturation_exponent_n = 1.6
theta_min = 0.02
theta_max = 0.50
deep_theta_background = 0.12
deep_anchor_depth_m = 5.0
upper_depth_m = 1.0
depth_ylim = (-90.0, 5.0)
delta_theta_clim = (-0.08, 0.08)

selected_labels = [
    '2022-03-26 00:30',
    '2022-04-20 06:30',
    '2022-05-02 06:30',
    '2022-05-12 18:30',
]

tmc_colors = {
    'TMC1': '#0072B2',
    'TMC2': '#E69F00',
    'TMC4': '#009E73',
    'TMC5': '#D55E00',
    'TMC6': '#CC79A7',
}

def savefig(fig, filename):
    if save_figures:
        out = result_dir / filename
        fig.savefig(out, dpi=figure_dpi, bbox_inches='tight')
        print('saved:', out)

print('project_root =', project_root)
print('result_dir   =', result_dir)



## 1) Load corrected resistivity, mesh and sensor data


In [ ]:
rho_corrected_file = result_dir / 'final_models_temperature_corrected.npy'
if not rho_corrected_file.exists():
    raise FileNotFoundError('Run 5_temperature_correction_real_resistivity.ipynb first.')
rho = np.load(rho_corrected_file)
coverage_mask = np.load(result_dir / 'coverage_mask.npy').astype(bool)

used_lines = (result_dir / 'used_data_files.txt').read_text().splitlines()
time_labels = [line.split('	')[0] for line in used_lines]
ert_times = pd.to_datetime(time_labels)
measurement_times_days = np.asarray((ert_times - ert_times[0]).total_seconds() / 86400.0, dtype=float)
label_to_index = {label: idx for idx, label in enumerate(time_labels)}
selected_indices = [label_to_index[label] for label in selected_labels if label in label_to_index]
if len(selected_indices) != len(selected_labels):
    missing = sorted(set(selected_labels).difference(label_to_index))
    raise ValueError(f'Selected labels not found: {missing}')

with np.load(result_dir / 'timelapse_inversion_mesh.npz') as mesh_data:
    nodes = np.asarray(mesh_data['nodes'], dtype=float)
    cells = np.asarray(mesh_data['cells'], dtype=np.int32)
    elec_x = np.asarray(mesh_data['elec_x'], dtype=float)
    elec_z = np.asarray(mesh_data['elec_z'], dtype=float)
    x_nodes = np.asarray(mesh_data['x_nodes'], dtype=float)
    z_top = np.asarray(mesh_data['z_top'], dtype=float)

cell_centers = nodes[cells].mean(axis=1)
cell_x = cell_centers[:, 0]
surface_at_center = np.interp(cell_x, x_nodes, z_top)
cell_depth = np.maximum(surface_at_center - cell_centers[:, 1], 0.0)
visible_cells = (~coverage_mask) & np.isfinite(rho[:, 0]) & (cell_depth >= 0.0)

sensor_locations = pd.read_csv(result_dir / 'real_sensor_locations_projected.csv')
sensor_ts = pd.read_csv(result_dir / 'real_sensor_timeseries_at_ert_times.csv')
sensor_ts['DateTime'] = pd.to_datetime(sensor_ts['DateTime'])

print('rho shape:', rho.shape)
print('sensor rows:', sensor_ts.shape[0])
print('selected:', [(i, time_labels[i]) for i in selected_indices])


## 2) Build a simple baseline theta0 model from TMC data


In [ ]:
sensor_depths = np.array([0.1, 0.3, 0.6, 0.8], dtype=float)
stations = sorted(sensor_ts['station'].unique())
station_names = []
station_x = []
station_theta0_profiles = []

for station in stations:
    loc = sensor_locations.loc[sensor_locations['ID'].eq(station)]
    if loc.empty:
        continue
    row0 = sensor_ts.loc[(sensor_ts['station'].eq(station)) & (sensor_ts['DateTime'].eq(ert_times[0]))]
    if row0.empty:
        continue

    theta_values = []
    for depth in sensor_depths:
        value = float(row0[f'MC_{depth:.1f}m'].iloc[0])
        theta_values.append(value)
    theta_values = np.asarray(theta_values, dtype=float)

    # Zero/negative MC values in this data set indicate sensor dropouts, not physical water content.
    if not np.all(np.isfinite(theta_values)) or np.any(theta_values <= 0.0):
        continue

    station_names.append(station)
    station_x.append(float(loc['profile_x_m'].iloc[0]))
    station_theta0_profiles.append(theta_values)

station_x = np.asarray(station_x, dtype=float)
station_theta0_profiles = np.asarray(station_theta0_profiles, dtype=float)
order = np.argsort(station_x)
station_x = station_x[order]
station_names = [station_names[i] for i in order]
station_theta0_profiles = station_theta0_profiles[order]

# Add a deep background anchor so shallow theta0 does not control deep cells.
theta_depths = np.r_[sensor_depths, deep_anchor_depth_m]
station_theta0_profiles = np.column_stack([
    station_theta0_profiles,
    np.full(station_theta0_profiles.shape[0], deep_theta_background, dtype=float),
])

theta0 = np.full(rho.shape[0], deep_theta_background, dtype=float)
clipped_depth = np.minimum(cell_depth, deep_anchor_depth_m)
station_cell_theta = np.vstack([
    np.interp(clipped_depth, theta_depths, profile, left=profile[0], right=deep_theta_background)
    for profile in station_theta0_profiles
])

for cell_idx in range(rho.shape[0]):
    theta0[cell_idx] = np.interp(
        cell_x[cell_idx],
        station_x,
        station_cell_theta[:, cell_idx],
        left=station_cell_theta[0, cell_idx],
        right=station_cell_theta[-1, cell_idx],
    )

theta0 = np.clip(theta0, theta_min, theta_max)

np.save(result_dir / 'theta0_sensor_interpolated.npy', theta0)
print('theta0 min/max/mean:', float(theta0.min()), float(theta0.max()), float(theta0.mean()))
pd.DataFrame(
    station_theta0_profiles[:, :4],
    index=station_names,
    columns=[f'{d:.1f}m' for d in sensor_depths],
)


## 3) Relative Archie estimate of theta(t) and Delta theta(t)


In [ ]:
rho0 = rho[:, 0]
delta_log_rho = np.log(rho) - np.log(rho0[:, None])
theta = theta0[:, None] * np.exp(-delta_log_rho / saturation_exponent_n)
theta = np.clip(theta, theta_min, theta_max)
delta_theta = theta - theta0[:, None]

np.save(result_dir / 'theta_relative_archie_estimate.npy', theta)
np.save(result_dir / 'delta_theta_relative_archie_estimate.npy', delta_theta)
relative_archie_summary = {
    'method': 'postprocess_relative_archie_from_temperature_corrected_resistivity',
    'saturation_exponent_n': float(saturation_exponent_n),
    'theta_min': float(theta_min),
    'theta_max': float(theta_max),
    'deep_theta_background': float(deep_theta_background),
    'deep_anchor_depth_m': float(deep_anchor_depth_m),
    'baseline_time': time_labels[0],
    'theta0_min': float(theta0.min()),
    'theta0_max': float(theta0.max()),
    'delta_theta_min': float(delta_theta.min()),
    'delta_theta_max': float(delta_theta.max()),
}
(result_dir / 'relative_archie_delta_theta_summary.json').write_text(json.dumps(relative_archie_summary, indent=2), encoding='utf-8')
relative_archie_summary


## 4) Plot theta0 and selected Delta theta sections


In [ ]:
def add_same_height_colorbar(fig, ax, mappable, label=None, title=None, size='4%', pad=0.04):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size=size, pad=pad)
    cbar = fig.colorbar(mappable, cax=cax)
    if label is not None:
        cbar.set_label(label)
    if title is not None:
        cbar.ax.set_title(title, pad=4)
    cbar.ax.tick_params(labelsize=10)
    return cbar

def format_section_axis(ax, *, show_xlabel=True, show_ylabel=True):
    ax.plot(elec_x, elec_z, color='black', lw=1.1, zorder=5)
    ax.set_xlim(float(elec_x.min()), float(elec_x.max()))
    ax.set_ylim(*depth_ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(False)
    if show_xlabel:
        ax.set_xlabel('Distance (m)')
    else:
        ax.tick_params(axis='x', labelbottom=False)
    if show_ylabel:
        ax.set_ylabel('Relative elevation (m)')
    else:
        ax.tick_params(axis='y', labelleft=False)

def plot_cell_model(ax, values, *, cmap, norm, mask=None):
    values = np.asarray(values, dtype=float).ravel()
    visible = np.isfinite(values)
    if mask is not None:
        visible &= ~np.asarray(mask, dtype=bool).ravel()
    pc = PolyCollection(nodes[cells][visible], array=values[visible], cmap=cmap, norm=norm, edgecolors='none')
    ax.add_collection(pc)
    return pc


In [ ]:
fig, ax = plt.subplots(figsize=(10.8, 4.2))
pc = plot_cell_model(ax, theta0, cmap='YlGnBu', norm=plt.Normalize(vmin=theta_min, vmax=0.35), mask=coverage_mask)
format_section_axis(ax)
ax.set_title('Baseline water content theta0 from TMC interpolation')
cbar = add_same_height_colorbar(fig, ax, pc, label='theta0 (V/V)')
fig.tight_layout()
savefig(fig, 'real_theta0_sensor_interpolated.png')
plt.show()


In [ ]:
delta_norm = TwoSlopeNorm(vmin=delta_theta_clim[0], vcenter=0.0, vmax=delta_theta_clim[1])
fig, axes = plt.subplots(1, len(selected_indices), figsize=(16.8, 4.2), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
last_pc = None
for col, (ax, idx) in enumerate(zip(axes, selected_indices, strict=True)):
    last_pc = plot_cell_model(ax, delta_theta[:, idx], cmap='BrBG', norm=delta_norm, mask=coverage_mask)
    format_section_axis(ax, show_xlabel=True, show_ylabel=(col == 0))
    ax.set_title(time_labels[idx])
fig.subplots_adjust(left=0.055, right=0.885, bottom=0.18, top=0.86, wspace=0.10)
if last_pc is not None:
    cax = fig.add_axes([0.905, 0.25, 0.012, 0.52])
    cbar = fig.colorbar(last_pc, cax=cax)
    cbar.set_label('Delta theta estimate (V/V)')
savefig(fig, 'real_delta_theta_relative_archie_selected_times.png')
plt.show()


## 5) Compare ERT-estimated Delta theta to TMC sensors


In [ ]:
# Extract the ERT estimate around each sensor using a small local cell average.
# This is more stable than taking one nearest cell, because the ERT model cell centers
# are much coarser than the TEROS sensor installation depths.
extract_dx_m = 7.5
extract_depth_half_window_m = 0.75
fallback_dx_m = 15.0
fallback_depth_half_window_m = 1.5

def local_visible_cells(x_target: float, depth_target: float) -> np.ndarray:
    mask = (
        visible_cells
        & (np.abs(cell_x - x_target) <= extract_dx_m)
        & (np.abs(cell_depth - depth_target) <= extract_depth_half_window_m)
    )
    ids = np.flatnonzero(mask)
    if ids.size > 0:
        return ids

    mask = (
        visible_cells
        & (np.abs(cell_x - x_target) <= fallback_dx_m)
        & (np.abs(cell_depth - depth_target) <= fallback_depth_half_window_m)
    )
    ids = np.flatnonzero(mask)
    if ids.size > 0:
        return ids

    candidates = np.flatnonzero(visible_cells)
    distance = ((cell_x[candidates] - x_target) / fallback_dx_m) ** 2 + ((cell_depth[candidates] - depth_target) / fallback_depth_half_window_m) ** 2
    return np.asarray([int(candidates[np.argmin(distance)])], dtype=int)

comparison_rows = []
for station in sorted(sensor_ts['station'].unique()):
    loc = sensor_locations.loc[sensor_locations['ID'].eq(station)]
    if loc.empty:
        continue

    x_target = float(loc['profile_x_m'].iloc[0])
    station_table = sensor_ts.loc[sensor_ts['station'].eq(station)].sort_values('DateTime')
    if len(station_table) != rho.shape[1]:
        raise ValueError(f'{station}: sensor/ERT time-axis length mismatch.')

    for depth in sensor_depths:
        sensor_col = f'MC_{depth:.1f}m'
        if sensor_col not in station_table.columns:
            continue

        sensor_values = station_table[sensor_col].to_numpy(dtype=float)
        sensor_values = np.where(sensor_values > 0.0, sensor_values, np.nan)
        if not np.isfinite(sensor_values[0]):
            continue
        sensor_delta = sensor_values - sensor_values[0]

        cell_ids = local_visible_cells(x_target, depth)
        ert_delta = np.nanmean(delta_theta[cell_ids, :], axis=0)
        theta0_local = float(np.nanmean(theta0[cell_ids]))

        for time_idx, timestamp in enumerate(ert_times):
            comparison_rows.append({
                'station': station,
                'depth_m': depth,
                'DateTime': timestamp,
                'day_since_baseline': measurement_times_days[time_idx],
                'profile_x_m': x_target,
                'n_cells_used': int(cell_ids.size),
                'mean_cell_x_m': float(np.nanmean(cell_x[cell_ids])),
                'mean_cell_depth_m': float(np.nanmean(cell_depth[cell_ids])),
                'theta0_ert_local': theta0_local,
                'delta_theta_sensor': sensor_delta[time_idx],
                'delta_theta_ert_estimate': ert_delta[time_idx],
            })

comparison = pd.DataFrame(comparison_rows)
comparison_file = result_dir / 'relative_archie_delta_theta_sensor_comparison.csv'
comparison.to_csv(comparison_file, index=False)
print('saved:', comparison_file)
comparison.head()


In [ ]:
# Time-series comparison, one panel per depth.
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2), sharex=True, sharey=True)
axes = axes.ravel()
for ax, depth in zip(axes, sensor_depths, strict=True):
    sub_depth = comparison.loc[np.isclose(comparison['depth_m'], depth)]
    for station in sorted(sub_depth['station'].unique()):
        sub = sub_depth.loc[sub_depth['station'].eq(station)].sort_values('DateTime')
        color = tmc_colors.get(station, 'tab:gray')
        ax.plot(sub['day_since_baseline'], sub['delta_theta_sensor'], color=color, lw=1.8, label=f'{station} sensor')
        ax.plot(sub['day_since_baseline'], sub['delta_theta_ert_estimate'], color=color, lw=1.5, ls='--', alpha=0.9, label=f'{station} ERT')
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.7)
    ax.set_title(f'Depth {depth:.1f} m')
    ax.set_xlabel('Days since baseline')
    ax.set_ylabel('Delta theta (V/V)')
    ax.grid(alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
# Keep one compact legend outside; repeated labels are intentional enough to show line style.
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=True)
fig.suptitle('Relative-Archie ERT estimate vs TMC soil-moisture change', y=1.02)
fig.tight_layout()
savefig(fig, 'real_delta_theta_ert_vs_sensor_timeseries.png')
plt.show()


In [ ]:
# Scatter and metrics across all stations/depths/times.
valid = comparison[['station', 'depth_m', 'delta_theta_sensor', 'delta_theta_ert_estimate']].replace([np.inf, -np.inf], np.nan).dropna()
residual = valid['delta_theta_ert_estimate'].to_numpy() - valid['delta_theta_sensor'].to_numpy()
rmse = float(np.sqrt(np.mean(residual ** 2)))
mae = float(np.mean(np.abs(residual)))
corr = float(np.corrcoef(valid['delta_theta_sensor'], valid['delta_theta_ert_estimate'])[0, 1]) if len(valid) > 2 else np.nan

metric_rows = []
for (station, depth), sub in valid.groupby(['station', 'depth_m']):
    sub_res = sub['delta_theta_ert_estimate'].to_numpy() - sub['delta_theta_sensor'].to_numpy()
    sub_corr = np.nan
    if len(sub) > 2 and np.nanstd(sub['delta_theta_sensor']) > 0 and np.nanstd(sub['delta_theta_ert_estimate']) > 0:
        sub_corr = float(np.corrcoef(sub['delta_theta_sensor'], sub['delta_theta_ert_estimate'])[0, 1])
    metric_rows.append({
        'station': station,
        'depth_m': float(depth),
        'rmse_delta_theta': float(np.sqrt(np.mean(sub_res ** 2))),
        'mae_delta_theta': float(np.mean(np.abs(sub_res))),
        'bias_delta_theta': float(np.mean(sub_res)),
        'correlation': sub_corr,
        'n_points': int(len(sub)),
    })

metrics_by_sensor = pd.DataFrame(metric_rows).sort_values(['station', 'depth_m'])
metrics_file = result_dir / 'relative_archie_delta_theta_sensor_metrics.csv'
metrics_by_sensor.to_csv(metrics_file, index=False)

fig, ax = plt.subplots(figsize=(5.8, 5.4))
for depth, sub in valid.groupby('depth_m'):
    ax.scatter(
        sub['delta_theta_sensor'],
        sub['delta_theta_ert_estimate'],
        s=22,
        alpha=0.62,
        label=f'{depth:.1f} m',
    )
lim = max(abs(valid['delta_theta_sensor']).max(), abs(valid['delta_theta_ert_estimate']).max(), 0.02)
ax.plot([-lim, lim], [-lim, lim], color='black', lw=1.1)
ax.axhline(0.0, color='black', lw=0.7, alpha=0.5)
ax.axvline(0.0, color='black', lw=0.7, alpha=0.5)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Sensor Delta theta (V/V)')
ax.set_ylabel('ERT estimated Delta theta (V/V)')
ax.set_title('Delta theta comparison at TMC locations')
ax.text(
    0.04,
    0.96,
    f'RMSE={rmse:.3f}\nMAE={mae:.3f}\nr={corr:.2f}',
    transform=ax.transAxes,
    va='top',
    ha='left',
    bbox=dict(facecolor='white', alpha=0.85, edgecolor='0.7'),
)
ax.legend(title='Depth', loc='lower right', frameon=True)
ax.grid(alpha=0.25)
fig.tight_layout()
savefig(fig, 'real_delta_theta_ert_vs_sensor_scatter.png')
plt.show()

metrics = {
    'rmse_delta_theta': rmse,
    'mae_delta_theta': mae,
    'correlation': corr,
    'n_points': int(len(valid)),
    'saturation_exponent_n': float(saturation_exponent_n),
}
metrics_json = result_dir / 'relative_archie_delta_theta_sensor_metrics.json'
metrics_json.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('saved:', metrics_file)
print('saved:', metrics_json)
metrics


## Output files

This notebook saves:

- `theta0_sensor_interpolated.npy`
- `theta_relative_archie_estimate.npy`
- `delta_theta_relative_archie_estimate.npy`
- `relative_archie_delta_theta_summary.json`
- `relative_archie_delta_theta_sensor_comparison.csv`
- `relative_archie_delta_theta_sensor_metrics.csv`
- `relative_archie_delta_theta_sensor_metrics.json`
- figure PNGs for `theta0`, `Delta theta`, and ERT-vs-sensor comparison.
